# Population Omega

Using the [human mortality database](https://www.mortality.org/Country/Country?cntr=USA) mortality rates, calculating the GLM omega rates

The data does include female, male, total qx from 1933 - 2021

A simple logistic regression will not fit the curve perfectly as there can be non-linear relationships with the shape of the curve. A spline is a good method to be able to model non-linear relationship as the mortality rates generally tend to accelerate and decelerate at different ages. It is recommended to use 3 degrees to represent gradual changes and somewhere between 3-5 knots.

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import polars as pl

os.chdir("../../")
sys.path.insert(0, os.getcwd())

In [2]:
from morai import models
from morai.experience import charters, tables
from morai.forecast import metrics, preprocessors
from morai.utils import custom_logger, helpers

In [3]:
# default is "plotly_mimetype+notebook", however that takes up space.
# "plotly_mimetype+notebook_connected" seems to save space
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Data

In [4]:
# importing data
colspecs = [(2, 7), (15, 20), (31, 40), (46, 56), (62, 72)]
colnames = ["observation_year", "attained_age", "female", "male", "qx_raw"]
hmd_usa_df = pd.read_fwf(
    "tests/files/forecast/models/hmd_usa_qx.txt",
    skiprows=3,
    colspecs=colspecs,
    names=colnames,
)
hmd_usa_df = hmd_usa_df[["observation_year", "attained_age", "qx_raw"]]
hmd_usa_df.rename(columns={"qx_raw": "qx_raw"}, inplace=True)
hmd_usa_df.loc[hmd_usa_df["attained_age"] == "110+", "attained_age"] = 110
hmd_usa_df["attained_age"] = hmd_usa_df["attained_age"].astype(int)
hmd_usa_df["qx_raw"] = hmd_usa_df["qx_raw"].astype(float)
hmd_usa_df = hmd_usa_df[hmd_usa_df["observation_year"] >= 1950]

In [5]:
preprocess_dict = preprocessors.preprocess_data(
    hmd_usa_df,
    feature_dict={
        "target": ["qx_raw"],
        "weight": [],
        "passthrough": [
            "observation_year",
            "attained_age",
        ],
    },
    standardize=False,
    add_constant=True,
)

X = preprocess_dict["X"]
y = preprocess_dict["y"]
weights = preprocess_dict["weights"]
mapping = preprocess_dict["mapping"]

 2025-05-12 00:58:26 | morai.forecast.preprocessors | INFO     | model target: ['qx_raw'] 
 2025-05-12 00:58:26 | morai.forecast.preprocessors | INFO     | adding a constant column to the data 
 2025-05-12 00:58:26 | morai.forecast.preprocessors | INFO     | passthrough - (generally numeric): ['attained_age', 'observation_year', 'constant'] 


# Models

## GLM

In [6]:
GLM = models.base.GLM()
GLM.fit(X, y)

 2025-05-12 00:58:26 | morai.models.base | INFO     | fiting the model 
 2025-05-12 00:58:26 | morai.models.base | INFO     | setup GLM model with statsmodels and <statsmodels.genmod.families.family.Binomial object at 0x0000021DBCB18CE0> family... 


In [7]:
hmd_usa_df["glm"] = GLM.predict(X)

## GAM

In [8]:
GAMR = models.gam.GAMR()
spline_dict = {
    "attained_age": {"bs":"tp"},
}

In [9]:
GAMR.setup_model(X=X, y=y, weights=weights, spline_dict=spline_dict)

 2025-05-12 00:58:26 | morai.models.gam | INFO     | setup GAM model with mgcv and `quasibinomial` distribution with `logit` link 
 2025-05-12 00:58:31 | morai.models.gam | INFO     | formula: 
 2025-05-12 00:58:31 | morai.models.gam | INFO     | > qx_raw ~ s(`attained_age`, bs='tp')+`constant`+`observation_year` 
 2025-05-12 00:58:31 | morai.models.gam | INFO     | fitting the model: 
 2025-05-12 00:58:31 | morai.models.gam | INFO     | > model <- bam(formula, data=data, weights=weights, family=family, drop.intercept=TRUE)` 
 2025-05-12 00:58:32 | morai.models.gam | INFO     | model fitted 


In [10]:
print(GAMR.summary(expand=True))

Generalized Additive Model (mgcv) Summary
Family                : quasibinomial
Link                  : logit
Number of Observations: 7992
Adjusted R-squared    : 0.932
Deviance Explained    : 96.4%
Scale Estimate        : 0.007575
fREML                 : -8128

Formula:
qx_raw ~ s(`attained_age`, bs='tp')+`constant`+`observation_year`

Parametric Coefficients:
constant           -12.138546
observation_year     0.003905

Smooth Terms:
                      edf    Ref.df            F  p-value     alpha
s(attained_age)  8.943415  8.998268  9835.265518      0.0  0.000032

Expanded Smooth Terms:
                 name       coef
2   s(attained_age).1 -13.643987
3   s(attained_age).2   4.259300
4   s(attained_age).3   4.608488
5   s(attained_age).4  -1.775516
6   s(attained_age).5  -1.791876
7   s(attained_age).6   2.036487
8   s(attained_age).7  -1.715768
9   s(attained_age).8  -1.227997
10  s(attained_age).9  -6.058854


In [11]:
hmd_usa_df["gam"] = GAMR.predict(X)

 2025-05-12 00:58:32 | morai.models.gam | INFO     | predicted rates 


# Compare

In [12]:
charters.compare_rates(
    df=hmd_usa_df,
    x_axis="attained_age",
    rates=["qx_raw", "glm", "gam"],
    display=True,
)

In [13]:
charters.compare_rates(
    df=hmd_usa_df[hmd_usa_df['observation_year']>=2000],
    x_axis="attained_age",
    rates=["qx_raw", "glm", "gam"],
    display=True,
)